In [1]:
import pandas as pd
import numpy as np
import math

In [2]:
ratings_columns_used = ['user', 'item', 'rating', 'timestamp']

In [3]:
data_path = 'datasets/ml_1m_one_task/'

In [4]:
df_train = pd.read_csv(data_path + 'train.csv')[ratings_columns_used]

C:\Users\admin\AppData\Local\Temp\ipykernel_2304\1817080535.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(data_path + 'train.csv')[ratings_columns_used]


In [5]:
df_test = pd.read_csv(data_path + 'test.csv')[ratings_columns_used]

In [6]:
df_test_cold = pd.read_csv(data_path + 'test_cold.csv')[ratings_columns_used]

In [7]:
df_users = pd.read_csv(data_path + 'user.csv')

In [8]:
df_items = pd.read_csv(data_path + 'item.csv')

In [9]:
df_items

,item,title,year,rate,released,genre,director,writer,actor,plot,poster
0,1,Toy Story,1995,G,22 Nov 1995,"Animation, Adventure, Comedy",John Lasseter,"John Lasseter (original story by), Pete Docter...","Tom Hanks, Tim Allen, Don Rickles, Jim Varney",A little boy named Andy loves to be in his roo...,https://m.media-amazon.com/images/M/MV5BMDU2ZW...
1,2,Jumanji,1995,PG,15 Dec 1995,"Adventure, Family, Fantasy",Joe Johnston,"Jonathan Hensleigh (screenplay), Greg Taylor (...","Robin Williams, Jonathan Hyde, Kirsten Dunst, ...",After being trapped in a jungle board game for...,https://m.media-amazon.com/images/M/MV5BZTk2Zm...
2,3,Grumpier Old Men,1995,PG-13,22 Dec 1995,"Comedy, Romance",Howard Deutch,"Mark Steven Johnson (characters), Mark Steven ...","Walter Matthau, Jack Lemmon, Sophia Loren, Ann...",Things don't seem to change much in Wabasha Co...,https://m.media-amazon.com/images/M/MV5BMjQxM2...
3,4,Waiting to Exhale,1995,R,22 Dec 1995,"Comedy, Drama, Romance",Forest Whitaker,"Terry McMillan (novel), Terry McMillan (screen...","Whitney Houston, Angela Bassett, Loretta Devin...",This story based on the best selling novel by ...,https://m.media-amazon.com/images/M/MV5BMTczMT...
4,5,Father of the Bride Part II,1995,PG,08 Dec 1995,"Comedy, Family, Romance",Charles Shyer,"Albert Hackett (screenplay ""Father's Little Di...","Steve Martin, Diane Keaton, Martin Short, Kimb...","In this sequel to ""Father of the Bride"", Georg...",https://m.media-amazon.com/images/M/MV5BOTEyNz...
...,...,...,...,...,...,...,...,...,...,...,...
3876,3948,Meet the Parents,2000,PG-13,06 Oct 2000,"Comedy, Romance",Jay Roach,"Greg Glienna, Mary Ruth Clarke, Greg Glienna (...","Robert De Niro, Ben Stiller, Teri Polo, Blythe...",A Jewish male nurse plans to ask his live-in g...,https://m.media-amazon.com/images/M/MV5BMGNlMG...
3877,3949,Requiem for a Dream,2000,R,15 Dec 2000,Drama,Darren Aronofsky,"Hubert Selby Jr. (based on the book by), Huber...","Ellen Burstyn, Jared Leto, Jennifer Connelly, ...",Sara Goldfarb (Ellen Burstyn) is a retired wid...,https://m.media-amazon.com/images/M/MV5BOTdiNz...
3878,3950,Tigerland,2000,R,24 May 2001,"Drama, War",Joel Schumacher,"Ross Klavan, Michael McGruther","Colin Farrell, Matthew Davis, Clifton Collins ...","In September 1971, a platoon of recruits arriv...",https://m.media-amazon.com/images/M/MV5BMzRlZD...
3879,3951,Two Family House,2000,R,27 Oct 2000,"Comedy, Drama, Romance",Raymond De Felitta,Raymond De Felitta,"Michael Rispoli, Kelly Macdonald, Kathrine Nar...","An unseen narrator looks back to 1956, on Stat...",https://images-na.ssl-images-amazon.com/images...


In [10]:
n_users = df_users['user'].max()
n_users = max(n_users, df_train['user'].max())
n_users = max(n_users, df_test['user'].max())
n_users = max(n_users, df_test_cold['user'].max())

In [11]:
n_items = df_items['item'].max()
n_items = max(n_items, df_train['item'].max())
n_items = max(n_items, df_test['item'].max())
n_items = max(n_items, df_test_cold['item'].max())

In [12]:
df_users = pd.merge(pd.DataFrame({'user':np.arange(n_users + 1)}), df_users, on = 'user', how = 'left')

In [13]:
df_items = pd.merge(pd.DataFrame({'item':np.arange(n_items + 1)}), df_items, on = 'item', how = 'left')

In [14]:
df_users

,user,gender,age,occupation,zipcode
0,0,NaN,NaN,NaN,NaN
1,1,M,50.0,0.0,66048
2,2,M,35.0,16.0,98133
3,3,M,25.0,20.0,95380
4,4,M,56.0,17.0,22657
...,...,...,...,...,...
6036,6036,M,1.0,10.0,28270
6037,6037,M,18.0,2.0,10010
6038,6038,M,18.0,2.0,95843
6039,6039,M,35.0,8.0,98674


In [15]:
user_features_columns = [column for column in df_users.columns if column != 'user']
count = 0
df_users['categories'] = ""
sep_used = ","
def processing_value(x, column_value_to_id, sep_used):
    if x == "":
        return x
    new_x = str(x).split(sep_used)
    values = [column_value_to_id[elem] for elem in new_x]
    values = sorted(list(set(values)))
    values = [str(elem) + ":1" for elem in values]
    return " ".join(values) + " "
def get_unique_column_values(values, sep_used):
    return list(set(sep_used.join(values).split(sep_used)))
for column in user_features_columns:
    df_users[column][pd.isnull(df_users[column])] = ""
    df_users[column] = df_users[column].astype(str)
    column_value_to_id = {str(value):index + count for index, value in enumerate(get_unique_column_values(df_users[column].unique(), sep_used)) if value != ""}
    df_users['categories'] += df_users[column].apply(lambda x : processing_value(x, column_value_to_id, sep_used))
    count += len(column_value_to_id)
    
    

C:\Users\admin\AppData\Local\Temp\ipykernel_2304\2264557782.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_users[column][pd.isnull(df_users[column])] = ""
C:\Users\admin\AppData\Local\Temp\ipykernel_2304\2264557782.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_users[column][pd.isnull(df_users[column])] = ""
C:\Users\admin\AppData\Local\Temp\ipykernel_2304\2264557782.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [16]:
item_features_columns = ["genre"]
count = 0
df_items['categories'] = ""
sep_used = ","
def processing_value(x, column_value_to_id, sep_used):
    if x == "":
        return x
    new_x = str(x).split(sep_used)
    values = [column_value_to_id[elem] for elem in new_x]
    values = sorted(list(set(values)))
    values = [str(elem) + ":1" for elem in values]
    return " ".join(values) + " "
def get_unique_column_values(values, sep_used):
    unique_values = list(set(sep_used.join(values).split(sep_used)))
    return unique_values
for column in item_features_columns:
    df_items[column][pd.isnull(df_items[column])] = ""
    df_items[column] = df_items[column].astype(str)
    column_value_to_id = {str(value):index + count for index, value in enumerate(get_unique_column_values(df_items[column].unique(), sep_used)) if value != ""}
    df_items['categories'] += df_items[column].apply(lambda x : processing_value(x, column_value_to_id, sep_used))
    df_items['categories'] = df_items['categories'].apply(lambda x : (x if x == "" else x[:-1]))
    count += len(column_value_to_id)
    
    

C:\Users\admin\AppData\Local\Temp\ipykernel_2304\1780873474.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_items[column][pd.isnull(df_items[column])] = ""


In [17]:
df_items

,item,title,year,rate,released,genre,director,writer,actor,plot,poster,categories
0,0,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,
1,1,Toy Story,1995.0,G,22 Nov 1995,"Animation, Adventure, Comedy",John Lasseter,"John Lasseter (original story by), Pete Docter...","Tom Hanks, Tim Allen, Don Rickles, Jim Varney",A little boy named Andy loves to be in his roo...,https://m.media-amazon.com/images/M/MV5BMDU2ZW...,1:1 2:1 16:1
2,2,Jumanji,1995.0,PG,15 Dec 1995,"Adventure, Family, Fantasy",Joe Johnston,"Jonathan Hensleigh (screenplay), Greg Taylor (...","Robin Williams, Jonathan Hyde, Kirsten Dunst, ...",After being trapped in a jungle board game for...,https://m.media-amazon.com/images/M/MV5BZTk2Zm...,4:1 19:1 31:1
3,3,Grumpier Old Men,1995.0,PG-13,22 Dec 1995,"Comedy, Romance",Howard Deutch,"Mark Steven Johnson (characters), Mark Steven ...","Walter Matthau, Jack Lemmon, Sophia Loren, Ann...",Things don't seem to change much in Wabasha Co...,https://m.media-amazon.com/images/M/MV5BMjQxM2...,39:1 44:1
4,4,Waiting to Exhale,1995.0,R,22 Dec 1995,"Comedy, Drama, Romance",Forest Whitaker,"Terry McMillan (novel), Terry McMillan (screen...","Whitney Houston, Angela Bassett, Loretta Devin...",This story based on the best selling novel by ...,https://m.media-amazon.com/images/M/MV5BMTczMT...,18:1 39:1 44:1
...,...,...,...,...,...,...,...,...,...,...,...,...
3948,3948,Meet the Parents,2000.0,PG-13,06 Oct 2000,"Comedy, Romance",Jay Roach,"Greg Glienna, Mary Ruth Clarke, Greg Glienna (...","Robert De Niro, Ben Stiller, Teri Polo, Blythe...",A Jewish male nurse plans to ask his live-in g...,https://m.media-amazon.com/images/M/MV5BMGNlMG...,39:1 44:1
3949,3949,Requiem for a Dream,2000.0,R,15 Dec 2000,Drama,Darren Aronofsky,"Hubert Selby Jr. (based on the book by), Huber...","Ellen Burstyn, Jared Leto, Jennifer Connelly, ...",Sara Goldfarb (Ellen Burstyn) is a retired wid...,https://m.media-amazon.com/images/M/MV5BOTdiNz...,43:1
3950,3950,Tigerland,2000.0,R,24 May 2001,"Drama, War",Joel Schumacher,"Ross Klavan, Michael McGruther","Colin Farrell, Matthew Davis, Clifton Collins ...","In September 1971, a platoon of recruits arriv...",https://m.media-amazon.com/images/M/MV5BMzRlZD...,22:1 43:1
3951,3951,Two Family House,2000.0,R,27 Oct 2000,"Comedy, Drama, Romance",Raymond De Felitta,Raymond De Felitta,"Michael Rispoli, Kelly Macdonald, Kathrine Nar...","An unseen narrator looks back to 1956, on Stat...",https://images-na.ssl-images-amazon.com/images...,18:1 39:1 44:1


In [18]:
def save_ratings(df, data_path, filename):
    df[ratings_columns_used].to_csv(data_path + filename, header=False, index = False)

In [19]:
def save_features(df, data_path, filename, columns_used):
    values = "\n".join((df[columns_used].astype(str) + " " + df['categories']).values)
    with open(data_path + filename, "w") as f:
        f.write(values)
        f.close()

In [20]:
saving_ratings_path = "ml_1m_for_dropout_net/warm/"

In [21]:
saving_features_path = "ml_1m_for_dropout_net/"

In [22]:
save_ratings(df_train, saving_ratings_path, "train.csv")
save_ratings(df_test_cold, saving_ratings_path, "test_cold_user.csv")
save_ratings(df_test, saving_ratings_path, "test_warm.csv")

In [23]:
df_items[['item']].to_csv(saving_ratings_path + 'test_cold_user_item_ids.csv', header=False, index = False)
df_items[['item']].to_csv(saving_ratings_path + 'test_warm_item_ids.csv', header=False, index = False)

In [24]:
save_features(df_items, saving_features_path, "item_features_0based.txt", 'item')

In [25]:
save_features(df_users, saving_features_path, "user_features_0based.txt", 'user')

In [26]:
pd.DataFrame({'user':df_test_cold['user'].unique()}).to_csv(saving_ratings_path + 'test_cold_user_ids.csv', index = False)

In [27]:
df_items['item'].nunique()

3953

In [28]:
len(df_items['item'])

3953

In [31]:
f"_{f"_{1}"}"

'_1'